# 02. 발주서 파싱 + 거래데이터 병합 (테스트용 노트북)

`02_parse_and_merge.py`와 동일한 로직을 셀 단위로 나눠서, 중간 결과를 눈으로 확인하며 실행할 수 있게 만든 버전입니다.

- 입력: `01_collect_and_clean_mail.ipynb`가 만든 `01-10_mail_clean_final.xlsx`
- 출력: 거래데이터 템플릿에 파싱 결과를 병합한 엑셀 파일

**실제 운영에 쓸 때는 `02_parse_and_merge.py`를 그대로 실행하세요.** 이 노트북은 파싱 로직을 바꾸거나 새 메일 포맷을 검증할 때 셀 단위로 확인하기 위한 용도입니다.

## 0. 준비: import 및 경로 설정

In [ ]:
import re
import datetime
from pathlib import Path
from collections import Counter
from openpyxl import load_workbook
import pandas as pd

BASE_DIR = Path.cwd()

# 01_collect_and_clean_mail.ipynb 의 결과물 (joagift/ 폴더에 함께 둠)
INPUT_MAIL = BASE_DIR / "output" / "01-10_mail_clean_final.xlsx"

# 실제 거래데이터 템플릿 파일 경로로 교체하세요.
# (예: C:\Users\USER\Desktop\김은지_업무파일\거래데이터_260220_기프트코_수정.xlsx)
INPUT_TMPL = Path(r"거래데이터_260220_기프트코_수정.xlsx")
INPUT_TMPL = Path(r"C:\Users\USER\Desktop\김은지_업무파일\거래데이터\기프트코_거래데이터_수집양식.xlsx")

OUTPUT_PATH = BASE_DIR / "output" / "joagift_transaction_data_test.xlsx"

## 1. 파싱 헬퍼 함수

In [2]:
def extract_between(text, start_key, end_keys):
    idx = text.find(start_key)
    if idx < 0:
        return ""
    start = idx + len(start_key)
    end = len(text)
    for ek in end_keys:
        ei = text.find(ek, start)
        if ei > 0 and ei < end:
            end = ei
    return text[start:end].strip()

In [3]:
ORG_KEYWORDS = [
    "학교", "대학교", "대학", "병원", "의원", "센터", "공사", "재단", "협회", "조합",
    "주식회사", "기업", "연구소", "농협", "신협", "금고", "은행", "지점",
    "구청", "시청", "군청", "행정복지센터", "지원청", "교육청", "소방서",
    "경찰서", "복지관", "어린이집", "유치원", "코리아", "Korea", "법인", "협동조합",
    "관광", "공단", "사업단", "재단법인", "의회", "협의회", "연합회", "문화원",
    "도서관", "박물관", "미술관", "사무소", "출장소", "관리소",
    "주유소", "새마을금고", "출판", "미디어", "광고", "기획", "엔지니어링",
    "솔루션", "테크", "시스템", "네트워크", "정보", "물산", "상사",
    "서비스", "컨설팅", "연구원", "요양원", "요양센터", "요양병원",
    "주민센터", "복지센터", "상담센터", "지원센터", "커뮤니케이션", "커뮤니티",
    "인더스트리", "파트너스", "어소시에이츠", "그룹", "홀딩스",
]

# 직함 접미어 — 뒤에 이게 붙으면 개인 이름이 아니라 직함 포함 수령자
TITLE_SUFFIXES = [
    "님", "씨", "팀장", "과장", "대리", "사원", "부장", "이사", "사장", "대표",
    "원장", "소장", "관장", "교수", "선생", "강사", "연구원", "연구사",
    "주무관", "담당자", "담당", "책임", "수석", "실장", "본부장", "센터장",
    "지점장", "지사장", "국장", "처장", "차장", "매니저", "MD", "md",
]

In [4]:
def extract_buyer_and_contact(recipient):
    """
    수령자 문자열 → (구매처명, 담당자명) 분리

    - "전남대학교 / 김명숙"              → ("전남대학교", "김명숙")
    - "교사 곽승철(세종과학예술영재학교)"  → ("세종과학예술영재학교", "교사 곽승철")
    - "국립국제교육원(우은숙님)"           → ("국립국제교육원", "우은숙")
    - "소나테크(주) 이주은"               → ("소나테크(주)", "이주은")
    - "고려대 의료원 차유심"              → ("고려대 의료원", "차유심")
    - "이영진 센터장님"                   → ("", "이영진 센터장님")  — 개인+직함
    - "서울특별시립 남부노인전문요양원"    → ("서울특별시립 남부노인전문요양원", "")
    - "신현주"                           → ("", "신현주")
    """
    r = recipient.strip()
    if not r:
        return "", ""

    # 패턴1: "기관 / 이름"
    m = re.match(r"^(.+?)\s*/\s*(.+)$", r)
    if m:
        return m.group(1).strip(), m.group(2).strip()

    # 패턴2: "(주)기관 이름" — (주)로 시작하는 법인 + 뒤에 짧은 한글 이름
    m = re.match(r"^(\(주\).+?)\s+([가-힣]{2,4})$", r)
    if m:
        return m.group(1).strip(), m.group(2).strip()

    # 패턴3: "기관명(주) 이름"
    m = re.match(r"^(.+?\(주\))\s+([가-힣]{2,4})$", r)
    if m:
        return m.group(1).strip(), m.group(2).strip()

    # 패턴4: "XXX(YYY)" — 괄호가 완전히 닫힌 경우
    m = re.match(r"^(.+?)\((.+)\)\s*$", r)
    if m:
        left   = m.group(1).strip()
        inside = m.group(2).strip()

        if inside in ("주", "유한"):
            return r, ""

        is_person_inside = (
            inside.endswith("님") or
            inside.endswith("씨") or
            (len(inside) <= 4 and re.match(r"^[가-힣]+$", inside)) or
            re.match(r"^\d", inside)
        )
        if is_person_inside:
            return left, re.sub(r"[님씨]$", "", inside)
        else:
            return inside, left

    # 패턴5: 괄호가 안 닫힌 경우 "XXX(YYY" — 괄호 안 내용으로 기관 판단
    m = re.match(r"^(.+?)\(([^)]+)$", r)
    if m:
        left   = m.group(1).strip()
        inside = m.group(2).strip()
        has_org_kw = any(kw in inside for kw in ORG_KEYWORDS)
        if has_org_kw or len(inside) >= 4:
            return inside, left
        return r, ""

    # 패턴6: 기관 키워드 포함 — 마지막 토큰이 짧은 한글 이름이면 분리
    for kw in ORG_KEYWORDS:
        if kw in r:
            tokens = r.split()
            last = tokens[-1]
            if (
                len(tokens) >= 2
                and len(last) <= 4
                and re.match(r"^[가-힣]+$", last)
                and kw not in last
            ):
                return " ".join(tokens[:-1]), last
            return r, ""

    # 나머지 → 개인 이름 (직함 포함도 그대로 담당자로)
    return "", r

In [ ]:
def normalize_text(text):
    """줄바꿈 제거 + 공백 정리 + 'T :' 같은 콜론 앞뒤 공백 표준화 (완성본 파서와 동일)"""
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"메일\s*:", "메일 :", text)
    text = re.sub(r"T\s*:", "T:", text)
    text = re.sub(r"F\s*:", "F:", text)
    return text


# 라벨이 실제 본문에 등장하는 순서는 메일마다 조금씩 다를 수 있어서,
# 정규식 하나씩 걸기보다 "라벨 위치를 찾아 정렬 → 라벨 사이 텍스트를 값으로 슬라이싱"하는 방식이 훨씬 안정적임
LABELS_IN_ORDER = [
    "품 명", "옵 션", "수 량", "단 가", "금 액", "부 가 세", "합 계",
    "인쇄문구", "발 송 일", "결 제 일", "도착주소", "배송수단",
    "업체담당", "담 당 자", "입금계좌", "비 고", "첨부파일",
]

PRINT_METHOD_KEYWORDS = [
    "전사인쇄", "실크인쇄", "실크", "레이저인쇄", "레이저", "UV인쇄", "UV",
    "자수", "박인쇄", "불박", "에폭시", "패드인쇄", "칼라인쇄", "옵셋인쇄", "풀컬러",
]


def make_remark(option_text, note_text):
    """비고 컬럼에 들어갈 텍스트 = 옵션 + 실제 '비 고' 라벨 섹션"""
    parts = []
    if option_text:
        parts.append(f"옵션: {option_text}")
    if note_text:
        parts.append(f"발주서비고: {note_text}")
    return " / ".join(parts)


def extract_tel(section_text):
    """'T:, 010-1234-5678' 처럼 첫 슬롯이 비고 콤마 뒤에 번호가 오는 경우까지 대응"""
    m = re.search(r"T:([^,\s]*),?\s*([\d\-]+)?", section_text)
    if not m:
        return ""
    t1 = (m.group(1) or "").strip()
    t2 = (m.group(2) or "").strip()
    return t1 if t1 else t2


def parse_any_order(body, subject):
    """발주서 본문에서 필드를 추출. 발주서가 아니면 None 반환.

    라벨("품 명", "옵 션" 등)이 있는 정식 서식과, 라벨이 없는 자유형 서식 둘 다 처리한다.
    """
    if not body or body == "None":
        return None
    b = str(body)

    # 발주서 여부 판단 (마커 or 상품코드) — 이 부분은 기존과 동일
    marker_pos = -1
    for marker in [
        "발 주 서 ( 주문번호",
        "발 주 서 (주)명성",
        "발주서 (주)명성",
        "(주)명성 귀하",
    ]:
        p = b.find(marker)
        if p >= 0:
            marker_pos = p
            break

    code_pos = b.find("-183-")
    if marker_pos < 0 and code_pos < 0:
        return None

    work_body_raw = b[marker_pos:] if marker_pos >= 0 else b[max(0, code_pos - 200):]
    text = normalize_text(work_body_raw)

    # ── 수령자 원문 / 구매처·담당자 분리 ───────
    recipient_raw = ""
    for pat in [r"\(수령자:([^\)]+)\)", r"수령자[:\s]+([^\s\(]+)"]:
        m = re.search(pat, text)
        if m:
            recipient_raw = m.group(1).strip()
            break
    if not recipient_raw:
        m = re.search(r"\(수령자:([^\)]+)\)", subject or "")
        if m:
            recipient_raw = m.group(1).strip()
    buyer_name, contact_person = extract_buyer_and_contact(recipient_raw)

    # ── 주문번호 ──────────────────────────────
    m = re.search(r"주문번호\s*:\s*(\d+)", text)
    order_num = m.group(1) if m else ""

    # ── 주문일(메일 제목에서) ──────────────────
    m = re.search(r"\[\s*(\d{4})-(\d{2})-(\d{2})", subject or "")
    order_date = None
    if m:
        try:
            order_date = datetime.datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        except Exception:
            pass

    is_labeled = any(x in text for x in ["품 명", "옵 션", "수 량", "단 가", "금 액"])

    product = ""
    option = ""
    qty = unit_price = amount = total = vat = None
    extra_cost = 0
    send_date = None
    tel = ""
    manager = ""
    print_text = ""
    remark_raw = ""

    if is_labeled:
        # ── 라벨 위치를 전부 찾아 정렬 → 라벨 사이를 값으로 슬라이싱 ──
        positions = []
        for label in LABELS_IN_ORDER:
            idx = text.find(label)
            if idx != -1:
                positions.append((idx, label))
        positions.sort()

        sections = {}
        for i, (start, label) in enumerate(positions):
            end = positions[i + 1][0] if i + 1 < len(positions) else len(text)
            sections[label] = text[start + len(label):end].strip(" :")

        product = sections.get("품 명", "")
        option = sections.get("옵 션", "")

        qm = re.search(r"([\d,]+)\s*EA", sections.get("수 량", ""), re.IGNORECASE)
        qty = int(qm.group(1).replace(",", "")) if qm else None

        pm = re.search(r"[￦₩]?\s*([\d,]+)", sections.get("단 가", ""))
        unit_price = int(pm.group(1).replace(",", "")) if pm else None

        money_section = sections.get("금 액", "")
        am = re.search(r"[￦₩]?\s*([\d,]+)", money_section)
        amount = int(am.group(1).replace(",", "")) if am else None

        em = re.search(r"\(([^)]+)\)", money_section)
        if em:
            for n in re.findall(r"[\d,]+원", em.group(1)):
                extra_cost += int(n.replace(",", "").replace("원", ""))

        vm = re.search(r"[￦₩]?\s*([\d,]+)", sections.get("부 가 세", ""))
        vat = int(vm.group(1).replace(",", "")) if vm else None

        tm = re.search(r"[￦₩]?\s*([\d,]+)", sections.get("합 계", ""))
        total = int(tm.group(1).replace(",", "")) if tm else None

        dm = re.search(r"(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일", sections.get("발 송 일", ""))
        if dm:
            try:
                send_date = datetime.datetime(int(dm.group(1)), int(dm.group(2)), int(dm.group(3)))
            except Exception:
                send_date = None

        tel = extract_tel(sections.get("도착주소", ""))

        manager = sections.get("담 당 자", "")

        print_text = sections.get("인쇄문구", "")
        if print_text in ["없음", "-"]:
            print_text = ""

        remark_raw = sections.get("비 고", "")

    else:
        # ── 라벨이 없는 자유형 서식: 숫자/기호 등장 순서로 추정 ──
        qty_match = re.search(r"(\d[\d,]*)\s*EA", text, re.IGNORECASE)
        money_matches = list(re.finditer(
            r"[￦₩]\s*[\d,]+(?:\s*\([^)]*(?:배송비|추가비)[^)]*\))?", text
        ))
        date_matches = list(re.finditer(r"\d{4}년\s*\d{1,2}월\s*\d{1,2}일", text))
        option_match = re.search(r"\[([^\]]+)\]", text)
        receiver_match = re.search(r"([가-힣]{2,6})\s*T:\s*(01[0-9]-\d{3,4}-\d{4})", text)

        if qty_match:
            product = text[:qty_match.start()].strip()
            qty = int(qty_match.group(1).replace(",", ""))

        def money_val(match_obj):
            mm = re.search(r"([\d,]+)", match_obj.group(0))
            return int(mm.group(1).replace(",", "")) if mm else None

        if len(money_matches) >= 1:
            unit_price = money_val(money_matches[0])
        if len(money_matches) >= 2:
            amount = money_val(money_matches[1])
        if len(money_matches) >= 4:
            total = money_val(money_matches[3])
        elif len(money_matches) >= 3:
            total = money_val(money_matches[-1])

        if option_match:
            option = option_match.group(1).strip()

        if date_matches:
            dm = re.search(r"(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일", date_matches[0].group(0))
            try:
                send_date = datetime.datetime(int(dm.group(1)), int(dm.group(2)), int(dm.group(3)))
            except Exception:
                send_date = None

        if receiver_match:
            if not recipient_raw:
                recipient_raw = receiver_match.group(1)
                buyer_name, contact_person = extract_buyer_and_contact(recipient_raw)
            tel = receiver_match.group(2)
        if not tel:
            tel = extract_tel(text)

        note_m = re.search(r"(발송일이 .*?)(?:메일\s*:\s*[\w\.-]+@[\w\.-]+\.\w+)", text)
        remark_raw = note_m.group(1).strip() if note_m else ""

    # ── 인쇄방법 (라벨/자유형 공통, 본문 전체에서 탐색) ─────
    print_method = ""
    for kw in PRINT_METHOD_KEYWORDS:
        if kw in text:
            print_method = kw
            break

    # ── 선물포장 (붙여쓰기/띄어쓰기 둘 다 대응) ─────────
    gift_wrap = "O" if re.search(r"선물\s*포장", option or "") or re.search(r"선물\s*포장", text[:500]) else ""

    if not product and not qty:
        return None

    remark = make_remark(option, remark_raw)

    return {
        "order_num":      order_num,
        "order_date":     order_date,
        "send_date":      send_date,
        "recipient_raw":  recipient_raw,
        "buyer_name":     buyer_name,
        "contact_person": contact_person,
        "manager":        manager,
        "tel":            tel,
        "product":        product,
        "unit_price":     unit_price,
        "qty":            qty,
        "amount":         amount,
        "extra_cost":     extra_cost or None,
        "total":          total,
        "vat":            vat,
        "print_text":     print_text,
        "print_method":   print_method,
        "gift_wrap":      gift_wrap,
        "option":         option,
        "remark":         remark,
    }

In [6]:
def get_mail_type(subject):
    """메일 제목에서 유형 판단: 원본 / Re(답장) / Fwd(전달)"""
    s = str(subject) if subject else ""
    if re.match(r"^(Re|RE)\s*:", s):
        return "Re(답장)"
    if re.match(r"^(FW|Fwd|FWD)\s*:", s, re.IGNORECASE):
        return "Fwd(전달)"
    if re.match(r"^(Re\s*:\s*)+(FW|Fwd)", s, re.IGNORECASE):
        return "Re(답장)"
    return "원본"

## 2. 메일 데이터 로드

In [7]:
wb_mail = load_workbook(INPUT_MAIL, read_only=True)
rows = list(wb_mail.active.iter_rows(values_only=True))
print(f"메일 총 {len(rows) - 1}건 로드")
rows[1][:2]  # 첫 데이터 행 미리보기 (제목, 본문 앞부분)

메일 총 2326건 로드


('조아기프트(주) 이메일 발주서 입니다. (수령자:신현주) [ 2023-02-17 11:15:21 ]',
 '발 주 서 ( 주문번호 : 5439674 ) 발 주 서 (주)명성 귀하 (수령자:신현주) 품 명 에스모도 무선충전 미니패드 206 -183-SMODO 206 옵 션 ★색상:화이트 / 전사인쇄 / 개별케이스★ 수 량 70EA ***급 당일출고부탁드립니다. 단 가 ￦5,000 **수정발주서:금액 수정해서 다시 보내드립니다 ^^ 금 액 ￦353,637 (배송비:3,637원 포함) 부 가 세 ￦35,364 합 계 ￦389,001 인쇄문구 조아-신현주님 [에스모도 무선충전 미니패드 206 ] 0217 발 송 일 2023년 2월 17일 결 제 일 2023년 2월 23일 입금자 : 조아기프트(주) 도착주소 서울 영등포구 은행로 17 (여의도동, 나이스1사옥) 7층 금융사업실 신현주 T:, 010-3203-2382 배송수단 택배(선불) ※ 보내는이 : 조아기프트(주) 표기요망 업체담당 강성철 이사 T:1588-8996, F:053-563-1211 담 당 자 영업팀 방문수MD (직통 :070-5092-5624) 입금계좌 농협 301-0023-3305-11 (주)명성 비 고 발송일이 2023-02-13 ~ 02-17일 까지 발주건의 세금계산서는 02-17일 1장으로 발행하여 02-21일 까지 보내주셔야 02-23일 에 결제가 가능합니다. * 발송일에 송장번호를 등록해 주셔야 결제됩니다. 메일 : tax@joagift.com 첨부파일 조아-신현주님 [에스모도 무선충전 미니패드 206 ] 0217.ai 상기와 같이 발주 합니다. 2023년 2월 17일')

## 3. 파싱 실행 (테스트)

In [8]:
parsed = []
for row in rows[1:]:
    result = parse_any_order(row[1], row[0])
    if result:
        result["mail_key"]  = row[2]
        result["uid"]       = row[3]
        result["mail_type"] = get_mail_type(row[0])
        parsed.append(result)

print(f"파싱 성공: {len(parsed)}건 / 전체 {len(rows) - 1}건")

파싱 성공: 1799건 / 전체 2326건


## 4. 파싱 결과 샘플 확인

실제 저장 전에 필드가 제대로 뽑혔는지 눈으로 확인하는 단계입니다.

In [9]:
df_preview = pd.DataFrame(parsed)
df_preview.head(10)

,order_num,order_date,send_date,recipient_raw,buyer_name,contact_person,manager,tel,product_code,product,...,extra_cost,total,vat,print_text,print_method,gift_wrap,option,mail_key,uid,mail_type
0,5439674,2023-02-17,2023-02-17,신현주,,신현주,영업팀 방문수MD (직통 :070-5092-5624),010-3203-2382,SMODO,에스모도 무선충전 미니패드 206,...,3637.0,389001.0,35364.0,조아-신현주님 [에스모도 무선충전 미니패드 206 ] 0217,전사인쇄,,★색상:화이트 / 전사인쇄 / 개별케이스★,&yHDFRK4w1QTSuA-_230217~|620810,620810,원본
1,5439869,2023-02-20,2023-02-23,교사 곽승철(세종과학예술영재학교,세종과학예술영재학교,교사 곽승철,영업팀 장미향과장 (직통 :070-5092-5616),010-3245-1079,SMODO,에스모도 멀티 고속충전 케이블 242,...,68184.0,1485092.0,135008.0,조아_곽승철님 세종과학예술영재학교 [에스모도 멀티 고속충전 케이블 242] 230217,칼라인쇄,,★인쇄 잘 부탁드립니다. 감사합니다.,&yHDFRK4w1QTSuA-_230217~|620887,620887,원본
2,5440210,2023-02-20,2023-02-22,강연민,,강연민,영업팀 성연숙MD (직통 :070-5092-5611),010-4604-8413,889,에스모도 미니 듀얼 일체형 보조배터리 10000mAh 889,...,25000.0,553000.0,0.0,"조아-강연민님 [에스모도 미니 듀얼 일체형 보조배터리 (10,000mAh)] 0220",UV인쇄,,★최대한 빠른 출고 요청드립니다★,&yHDFRK4w1QTSuA-_230217~|620899,620899,원본
3,5442983,2023-02-28,2023-02-28,문창훈,,문창훈,영업팀 이혜미팀장 (직통 :070-5092-5602),010-3303-9378,MS300,에스모도 무선 고속충전 보조배터리 10000mAh 300,...,3000.0,20200.0,1836.0,,인쇄없음,,,&yHDFRK4w1QTSuA-_230217~|621491,621491,원본
4,5444212,2023-03-06,2023-03-07,홍서아,,홍서아,영업팀 조미향MD (직통 :070-5092-5610),02-2260-2711,MS004,에스모도 6인치 탁상형 무선선풍기 004,...,30000.0,214000.0,0.0,조아-홍서아님[에스모도 6인치 탁상형 무선 선풍기 (3900mAh)],레이저인쇄,,,&yHDFRK4w1QTSuA-_230217~|621879,621879,원본
5,5445447,2023-03-07,2023-03-14,김창숙,,김창숙,영업팀 김경림MD (직통 :070-5092-5618),010-2054-8549,SMODO,에스모도 멀티 고속충전 케이블 242,...,25000.0,495000.0,0.0,조아_김창숙님 [에스모도 멀티 고속충전 케이블 242] 230307,전사인쇄,O,,&yHDFRK4w1QTSuA-_230217~|622032,622032,원본
6,5447058,2023-03-11,2023-03-13,안정현,,안정현,영업팀 김경화팀장 (직통 :070-5092-5627),010-4040-7989,SMODO,에스모도 멀티 고속충전 케이블 242,...,2727.0,7700.0,700.0,- 색상 : 화아트/인쇄없음/개별케이스,,,,&yHDFRK4w1QTSuA-_230217~|622358,622358,원본
7,5448971,2023-03-16,2023-03-21,JohnKim,,JohnKim,영업팀 황민경과장 (직통 :070-5092-5613),02-6478-8112,SMODO,에스모도 슬림핏 보조배터리 5000mAh 886,...,16368.0,2388065.0,217097.0,조아-JohnKim(brainai) [보조배터리] 0316,칼라인쇄,,,&yHDFRK4w1QTSuA-_230217~|622927,622927,원본
8,5451319,2023-03-22,2023-03-27,키오솔루션,키오솔루션,,영업팀 성연숙MD (직통 :070-5092-5611),02-2163-0091,SMODO,에스모도 고속 무선충전 펜꽂이 208A,...,10911.0,1530002.0,139091.0,조아-키오솔루션 [에스모도 고속 무선충전 펜꽂이] 0322,,,3/24일에 출고 가능하다면 부탁드립니다. 행사일 29일사용,&yHDFRK4w1QTSuA-_230217~|623384,623384,원본
9,5452613,2023-03-24,2023-03-30,이영진 센터장님,이영진 센터장님,,복지몰사업팀 성기영팀장 (직통 :070-5092-5635),010-8003-8529,MS004,에스모도 6인치 탁상형 무선선풍기 004,...,32733.0,1301006.0,118273.0,*택배 보내시는분:서울디지털인쇄협동조합 에스모도 6인치 탁상형 무선 선풍기 (390...,레이저인쇄,,,&yHDFRK4w1QTSuA-_230217~|623598,623598,원본


## 5. 메일유형 / 주문번호 중복 통계

In [10]:
order_num_counter = Counter(d["order_num"] for d in parsed if d["order_num"])
for d in parsed:
    onum = d["order_num"]
    d["dup_flag"] = f"중복({order_num_counter[onum]}건)" if onum and order_num_counter[onum] > 1 else ""

type_counts = Counter(d["mail_type"] for d in parsed)
dup_count = sum(1 for d in parsed if d["dup_flag"])

print(f"원본: {type_counts['원본']}건 / Re(답장): {type_counts['Re(답장)']}건 / Fwd(전달): {type_counts['Fwd(전달)']}건")
print(f"주문번호 중복 포함 행: {dup_count}건")

원본: 1647건 / Re(답장): 145건 / Fwd(전달): 7건
주문번호 중복 포함 행: 144건


## 6. 거래데이터 템플릿 병합 (실제 저장)

**주의**: 이 셀은 `INPUT_TMPL` 파일을 실제로 열어 기존 데이터 행(5행부터)을 지우고 새로 씁니다.
템플릿 경로가 맞는지, 원본 파일을 덮어쓰는 게 아니라 새 파일(`OUTPUT_PATH`)로 저장되는지 확인 후 실행하세요.

In [ ]:
wb_out = load_workbook(INPUT_TMPL)
ws_out = wb_out["거래데이터"]
for row_idx in range(ws_out.max_row, 4, -1):
    ws_out.delete_rows(row_idx)

ws_out.cell(1, 29, "메일유형")
ws_out.cell(1, 30, "주문번호중복")
ws_out.cell(1, 31, "메일키")
ws_out.cell(1, 32, "UID")
ws_out.cell(1, 33, "수령자원문")
ws_out.cell(1, 34, "조아기프트담당자(참고용)")

for i, d in enumerate(parsed, start=1):
    r = 4 + i
    vals = [
        i, d["order_num"], "조아기프트", d["order_date"], d["send_date"],
        "", "", d["buyer_name"], d["contact_person"], d["tel"],
        "", "", None, d["product"], "",
        "", "", "", d["unit_price"], "",
        d["qty"], d["amount"], d["extra_cost"], d["total"],
        d["print_method"], "", d["gift_wrap"], d["remark"],
        d["mail_type"], d["dup_flag"],
        d["mail_key"], d["uid"], d["recipient_raw"], d["manager"],
    ]
    for col, val in enumerate(vals, start=1):
        if val == "None" or val is None:
            val = None
        cell = ws_out.cell(row=r, column=col, value=val)
        if isinstance(val, datetime.datetime):
            cell.number_format = "YYYY-MM-DD"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
wb_out.save(OUTPUT_PATH)
print(f"저장 완료 → {OUTPUT_PATH}")